### Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [8]:
PROJECT_NAME = "cs172b-balance"

default_config = {
    "epochs": 5,
    "learning_rate": 0.01,
    "batch_size": 64,
    "regression_penalty": 0.01,
    "input_dim": 784,
    "hidden_dim": 128,
    "output_dim": 10,
}

sweep_config = {
    "method": "grid",
    "metric": {
        "name": "train_loss",
        "goal": "minimize"
    },
    "parameters": {
        "learning_rate": {
            "values": [0.1, 0.01, 0.001]
        },
        "batch_size": {
            "values": [16, 32, 64]
        },
        "regression_penalty": {
            "values": [0.0, 1e-4, 1e-3]
        }
    }
}

### Datasets

In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

### Models

In [10]:
class FNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### Training

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[device]: ", device)

def train(config=None):
    with wandb.init(project=PROJECT_NAME, config=default_config):
        config = wandb.config

        wandb.run.name = (
          f"lr={config.learning_rate:.0e}_"
          f"bs={config.batch_size}_"
          f"reg={config.regression_penalty:.0e}"
        )

        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
        test_loader  = DataLoader(test_dataset,  batch_size=config.batch_size, shuffle=False, pin_memory=True)

        model     = FNN(config.input_dim, config.hidden_dim, config.output_dim).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(model.parameters(), lr=config.learning_rate)

        for epoch in range(config.epochs):
          running_loss = 0
          for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()

            out  = model(x)
            loss = criterion(out, y)
            reg  = sum(p.pow(2).sum() for p in model.parameters())

            loss = loss + config.regression_penalty * reg
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        wandb.log({
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader)
        })

[device]:  cuda


In [12]:
sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, function=train)

Create sweep with ID: nkn3g7ia
Sweep URL: https://wandb.ai/lmacadar-uci/cs172b-balance/sweeps/nkn3g7ia


wandb: Agent Starting Run: icjb3kju with config:
wandb: 	batch_size: 16
wandb: 	learning_rate: 0.1
wandb: 	regression_penalty: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁
train_loss,▁
epoch,4
train_loss,0.05164


wandb: Agent Starting Run: r291h8ou with config:
wandb: 	batch_size: 16
wandb: 	learning_rate: 0.1
wandb: 	regression_penalty: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁
train_loss,▁
epoch,4
train_loss,0.09208


...

epoch,▁
train_loss,▁
epoch,4
train_loss,0.20179


wandb: Agent Starting Run: oetnjc6z with config:
wandb: 	batch_size: 16
wandb: 	learning_rate: 0.01
wandb: 	regression_penalty: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch,▁
train_loss,▁
epoch,4
train_loss,0.21506


wandb: Agent Starting Run: ahkmwaq2 with config:
wandb: 	batch_size: 16
wandb: 	learning_rate: 0.01
wandb: 	regression_penalty: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Ctrl + C detected. Stopping sweep.
